# NLP Group Assignment — Question 1
# Word Segmentation and POS Tagging (English + Spanish)

This notebook implements the full pipeline requested in Q1:

1. **Segmentation** — a trigram word language model + Viterbi (dynamic programming) to split a space-free string into words.
2. **POS tagging** — a trigram HMM (transition + emission probabilities) decoded with Viterbi.
3. **Morphology-aware tagging** — for Spanish, tags are extended with gender/number (e.g. `NOUN-Fem-Sg`) so agreement can be modelled and evaluated.
4. **Baselines** — greedy longest-match segmentation, and most-frequent-tag tagging.
5. **Evaluation** — accuracy, a confusion matrix, and an error-source breakdown that separates *segmentation-caused* tagging errors from *genuine* tagging errors.

**Languages:** English (Brown Corpus) and Spanish (UD Spanish-GSD).

**Data availability note:** this notebook first tries to load the real corpora (`nltk.corpus.brown`, and a locally-cloned `UD_Spanish-GSD`). If either is unavailable (no internet / not downloaded yet), it automatically falls back to a small embedded corpus so every cell below still runs end-to-end and you can see the whole pipeline work. **For your actual submission, run this with the real corpora downloaded** (see the setup cell) — the embedded corpora are only a tiny illustrative stand-in and will not give meaningful accuracy numbers.



## 0. Setup

In [18]:
import re, math, random, itertools
from collections import defaultdict, Counter

random.seed(42)
BOS = "<s>"           # sentence-boundary token, reused for both word- and tag-level n-grams

UNK_LOGPROB_PER_CHAR = -4.0
def unk_penalty(word):
    return UNK_LOGPROB_PER_CHAR * len(word)


In [19]:
import nltk 
nltk.download('brown')

/home/parth/college/venv/lib/python3.14/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/parth/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package brown to /home/parth/nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

## 1. Data loading

`load_english_data()` uses NLTK's Brown corpus (simplified/universal tagset for readability — swap
`tagset='universal'` for the native Brown tagset if you want the fine-grained `DT/JJ/NN/VBZ/...`
style tags shown in the assignment's example).

`load_spanish_data()` reads the cloned `UD_Spanish-GSD` CoNLL-U files (train/dev/test) and keeps
the UPOS tag plus the morphological feature dictionary (`Gender`, `Number`, ...) for each token —
this is what Part 3 (morphology-aware tagging) needs.

If neither the NLTK data nor the UD repo is found on disk, both loaders fall back to a small
embedded corpus (defined below) purely so the notebook is runnable offline for demonstration.

In [20]:
ENGLISH_FALLBACK = [
    [("the","DET"),("quick","ADJ"),("brown","ADJ"),("fox","NOUN"),("jumps","VERB"),
     ("over","ADP"),("the","DET"),("lazy","ADJ"),("dog","NOUN")],
    [("the","DET"),("dog","NOUN"),("barks","VERB"),("at","ADP"),("the","DET"),("cat","NOUN")],
    [("a","DET"),("quick","ADJ"),("cat","NOUN"),("runs","VERB"),("over","ADP"),("the","DET"),("lazy","ADJ"),("fox","NOUN")],
    [("she","PRON"),("eats","VERB"),("a","DET"),("green","ADJ"),("salad","NOUN")],
    [("the","DET"),("man","NOUN"),("saw","VERB"),("the","DET"),("dog","NOUN"),("with","ADP"),("a","DET"),("telescope","NOUN")],
    [("the","DET"),("cat","NOUN"),("sat","VERB"),("on","ADP"),("the","DET"),("mat","NOUN")],
]

SPANISH_FALLBACK = [
    [("la","DET",{"Gender":"Fem","Number":"Sing"}),("casa","NOUN",{"Gender":"Fem","Number":"Sing"}),
     ("roja","ADJ",{"Gender":"Fem","Number":"Sing"}),("es","AUX",{}),("grande","ADJ",{"Number":"Sing"})],
    [("el","DET",{"Gender":"Masc","Number":"Sing"}),("cielo","NOUN",{"Gender":"Masc","Number":"Sing"}),
     ("despejado","ADJ",{"Gender":"Masc","Number":"Sing"}),("es","AUX",{}),("azul","ADJ",{"Number":"Sing"})],
    [("mis","DET",{"Number":"Plur"}),("padres","NOUN",{"Gender":"Masc","Number":"Plur"}),
     ("pueden","VERB",{"Number":"Plur"}),("viajar","VERB",{})],
    [("las","DET",{"Gender":"Fem","Number":"Plur"}),("casas","NOUN",{"Gender":"Fem","Number":"Plur"}),
     ("rojas","ADJ",{"Gender":"Fem","Number":"Plur"}),("son","AUX",{}),("grandes","ADJ",{"Number":"Plur"})],
    [("el","DET",{"Gender":"Masc","Number":"Sing"}),("perro","NOUN",{"Gender":"Masc","Number":"Sing"}),
     ("negro","ADJ",{"Gender":"Masc","Number":"Sing"}),("corre","VERB",{})],
    [("la","DET",{"Gender":"Fem","Number":"Sing"}),("nina","NOUN",{"Gender":"Fem","Number":"Sing"}),
     ("pequena","ADJ",{"Gender":"Fem","Number":"Sing"}),("lee","VERB",{})],
]


_BROWN_TO_UNIVERSAL_PREFIXES = [
    # ordered longest-prefix-first; matched after stripping brown's '-TL/-HL/-NC' suffixes and '*'
    ("NP", "NOUN"), ("NR", "NOUN"), ("NN", "NOUN"),
    ("VB", "VERB"), ("DO", "VERB"), ("HV", "VERB"), ("BE", "VERB"), ("MD", "VERB"),
    ("JJ", "ADJ"),
    ("QL", "ADV"), ("RB", "ADV"), ("RN", "ADV"), ("RP", "PRT"), ("WRB", "ADV"),
    ("IN", "ADP"),
    ("CC", "CONJ"), ("CS", "CONJ"),
    ("WDT", "DET"), ("AT", "DET"), ("DT", "DET"), ("AP", "DET"), ("AB", "DET"),
    ("CD", "NUM"), ("OD", "NUM"),
    ("WPS", "PRON"), ("WPO", "PRON"), ("WP", "PRON"), ("PP", "PRON"), ("PN", "PRON"),
    ("TO", "PRT"),
    ("UH", "X"), ("FW", "X"), ("EX", "DET"), ("NIL", "X"),
]

def brown_tag_to_universal(tag):
    """Self-contained Brown -> Universal (coarse) tag mapping.
    Avoids depending on the (deprecated / download-flaky) nltk `universal_tagset` resource
    and the `tagset='universal'` kwarg to `tagged_sents()` — we just fetch raw Brown tags and
    map them ourselves."""
    t = tag.split('-')[0].split('+')[0].rstrip('*').rstrip('$')
    if not t or not t[0].isalpha():
        return "."
    for prefix, universal in _BROWN_TO_UNIVERSAL_PREFIXES:
        if t.startswith(prefix):
            return universal
    return "X"


def load_english_data():
    """Returns a list of sentences; each sentence is a list of (word, tag)."""
    try:
        import nltk
        from nltk.corpus import brown
        try:
            brown.tagged_sents()
        except LookupError:
            nltk.download('brown')
        sents = brown.tagged_sents()  # raw Brown tags; no tagset kwarg / no extra download
        data = [[(w.lower(), brown_tag_to_universal(t)) for w, t in s] for s in sents]
        print(f"[english] loaded {len(data)} sentences from NLTK Brown corpus")
        return data
    except Exception as e:
        print(f"[english] NLTK/Brown unavailable ({e}); using embedded fallback corpus")
        return ENGLISH_FALLBACK * 40  # repeated so the LMs have enough counts to be non-trivial


def parse_conllu(text):
    """Minimal CoNLL-U reader -> list of sentences of (form, upos, feats_dict)."""
    sents, cur = [], []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            if cur:
                sents.append(cur); cur = []
            continue
        if line.startswith('#'):
            continue
        cols = line.split('\t')
        if len(cols) < 10 or '-' in cols[0] or '.' in cols[0]:
            continue  # skip multiword-token / empty-node lines
        form, upos, feats_raw = cols[1], cols[3], cols[5]
        feats = {}
        if feats_raw != '_':
            for kv in feats_raw.split('|'):
                if '=' in kv:
                    k, v = kv.split('=', 1)
                    feats[k] = v
        cur.append((form.lower(), upos, feats))
    if cur:
        sents.append(cur)
    return sents


def load_spanish_data(path_prefix='./UD_Spanish-GSD/es_gsd-ud'):
    try:
        with open(f'{path_prefix}-train.conllu', encoding='utf-8') as f:
            data = parse_conllu(f.read())
        print(f"[spanish] loaded {len(data)} sentences from {path_prefix}-train.conllu")
        return data
    except Exception as e:
        print(f"[spanish] UD_Spanish-GSD not found locally ({e}); using embedded fallback corpus")
        return SPANISH_FALLBACK * 40


## 2. Morphology-aware tags (Part 3)

For Spanish, we build a composite tag such as `NOUN-Fem-Sg` from the UPOS tag plus the `Gender`
and `Number` morphological features in the UD annotation. Training the same trigram HMM on these
composite tags means the transition model now has to learn **agreement patterns**
(e.g. a `NOUN-Fem-Sg` is much more likely to be followed by an `ADJ-Fem-Sg` than an
`ADJ-Masc-Pl`).

In [21]:
GENDER_ABBR = {"Masc": "Masc", "Fem": "Fem", "Neut": "Neut"}
NUMBER_ABBR = {"Sing": "Sg", "Plur": "Pl"}

def build_morph_tag(upos, feats):
    """e.g. NOUN + {Gender:Fem, Number:Sing} -> 'NOUN-Fem-Sg'."""
    parts = [upos]
    if "Gender" in feats:
        parts.append(GENDER_ABBR.get(feats["Gender"], feats["Gender"]))
    if "Number" in feats:
        parts.append(NUMBER_ABBR.get(feats["Number"], feats["Number"]))
    return "-".join(parts)


def spanish_to_tagged(sents, morph=False):
    """Convert (form, upos, feats) sentences to (word, tag) sentences."""
    out = []
    for s in sents:
        if morph:
            out.append([(w, build_morph_tag(u, f)) for w, u, f in s])
        else:
            out.append([(w, u) for w, u, f in s])
    return out


## 3. Trigram language model (for segmentation scoring)

A generic trigram model over a stream of word tokens, with add-k smoothing and back-off to
bigram/unigram statistics when a trigram context hasn't been seen. This is the model that scores
candidate segmentations in Part 1.

In [22]:
class TrigramLM:
    """Trigram model over a token stream (words), with add-k smoothing and
    back-off to bigram/unigram estimates for unseen contexts."""

    def __init__(self, k=0.5):
        self.k = k
        self.uni = Counter()
        self.bi = Counter()
        self.tri = Counter()
        self.vocab = set()
        self.total = 0

    def train(self, sentences_of_words):
        for words in sentences_of_words:
            seq = [BOS, BOS] + list(words) + ["</s>"]
            for w in words:
                self.vocab.add(w)
            for i in range(len(seq)):
                self.uni[seq[i]] += 1
                self.total += 1
                if i >= 1:
                    self.bi[(seq[i-1], seq[i])] += 1
                if i >= 2:
                    self.tri[(seq[i-2], seq[i-1], seq[i])] += 1
        self.V = max(len(self.vocab), 1)

    def logprob(self, w, u=None, v=None):
        """log P(w | v, u) with back-off: trigram -> bigram -> unigram."""
        k = self.k
        if u is not None and v is not None:
            num = self.tri[(v, u, w)] + k
            den = self.bi[(v, u)] + k * self.V
            if den > 0:
                return math.log(num / den)
        if u is not None:
            num = self.bi[(u, w)] + k
            den = self.uni[u] + k * self.V
            if den > 0:
                return math.log(num / den)
        num = self.uni[w] + k
        den = self.total + k * self.V
        return math.log(num / max(den, 1e-12))


## 4. Part 1 — Word segmentation (trigram LM + Viterbi DP)

**Design note on tractability.** A literal trigram DP over word segmentations would need a state
of *(position, previous word, word-before-that)*, which blows up the state space to
`O(n · V²)`. Instead we use the standard **second-order Viterbi trick**: the DP state only needs
to be *(position, last word)* — `dp[j][last_word] = (best_score, prev_word, back_pointer)`. The
word *before* `prev_word` (needed to score the *next* trigram transition) is recovered lazily by
following the back-pointer chain into `dp[back_i][prev_word]`. This keeps the algorithm at
`O(n · max_word_len · |vocab_at_each_position|)` while still scoring every step with the full
trigram probability, and is exactly the same trick used for second-order Viterbi POS tagging in
Part 2 below.

We also implement the **greedy longest-match** baseline required in Part 4.

In [23]:
def viterbi_segment(chars, lm, vocab, max_word_len=15):
    """Second-order (trigram) Viterbi word segmentation over a DP trellis of
    character positions. Returns the best list of word tokens for `chars`."""
    n = len(chars)
    dp = [dict() for _ in range(n + 1)]
    dp[0][BOS] = (0.0, BOS, None)  # at position 0, "last word so far" = BOS

    for j in range(1, n + 1):
        best_for_word = {}
        start = max(0, j - max_word_len)
        for i in range(start, j):
            w = chars[i:j]
            if not dp[i]:
                continue
            in_vocab = w in vocab
            for u, (score_i, v, _back) in dp[i].items():
                lp = lm.logprob(w, u, v)
                if not in_vocab:
                    lp += unk_penalty(w)
                cand = score_i + lp
                if w not in best_for_word or cand > best_for_word[w][0]:
                    best_for_word[w] = (cand, u, i)
        dp[j] = best_for_word

    if not dp[n]:
        return list(chars)  # total failure fallback: character-by-character

    best_word = max(dp[n], key=lambda w: dp[n][w][0])
    words = []
    j, w = n, best_word
    while j > 0:
        sc, u, i = dp[j][w]
        words.append(chars[i:j])
        j, w = i, u
    words.reverse()
    return words


def greedy_longest_match_segment(chars, vocab, max_word_len=15):
    """Baseline: at each position, take the longest prefix that is in vocab."""
    n = len(chars)
    i, words = 0, []
    while i < n:
        matched = None
        for j in range(min(n, i + max_word_len), i, -1):
            if chars[i:j] in vocab:
                matched = chars[i:j]
                i = j
                break
        if matched is None:
            matched = chars[i]
            i += 1
        words.append(matched)
    return words


In [24]:
# quick sanity check with a tiny toy LM
_toy_sents = [["the","quick","brown","fox","jumps","over","the","lazy","dog"]]
_toy_lm = TrigramLM(k=0.3); _toy_lm.train(_toy_sents)
_toy_vocab = set(_toy_sents[0])
print(viterbi_segment("thequickbrownfoxjumpsoverthelazydog", _toy_lm, _toy_vocab, 12))
print(greedy_longest_match_segment("thequickbrownfoxjumpsoverthelazydog", _toy_vocab, 12))


['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']


## 5. Part 2 — Trigram HMM POS tagger (emission + transition, Viterbi)

Standard second-order HMM tagging:
- **Emission** `P(word | tag)`, add-k smoothed, with a simple unknown-word fallback.
- **Transition** `P(tag_i | tag_{i-1}, tag_{i-2})`, add-k smoothed.
- **Decoding**: trigram Viterbi over `(tag_{i-1}, tag_i)` pair-states — this is the classic
  formulation (unlike segmentation, the tag vocabulary is small, so we can afford to enumerate
  tag-pairs directly rather than needing the back-pointer trick above).

The **most-frequent-tag baseline** (Part 4) is also defined here.

In [25]:
class TrigramHMM:
    def __init__(self, k_trans=0.1, k_emit=0.1):
        self.k_trans, self.k_emit = k_trans, k_emit
        self.trans_tri = Counter()   # (t-2,t-1) -> t counts
        self.trans_bi = Counter()    # (t-2,t-1) counts
        self.emit = Counter()        # (tag, word) counts
        self.tag_count = Counter()
        self.word_count = Counter()
        self.tags = set()

    def train(self, tagged_sentences):
        for sent in tagged_sentences:
            tags = [BOS, BOS] + [t for _, t in sent]
            for _, t in sent:
                self.tags.add(t)
            for i in range(2, len(tags)):
                self.trans_tri[(tags[i-2], tags[i-1], tags[i])] += 1
                self.trans_bi[(tags[i-2], tags[i-1])] += 1
            for w, t in sent:
                self.emit[(t, w)] += 1
                self.tag_count[t] += 1
                self.word_count[w] += 1
        self.T = max(len(self.tags), 1)
        self.tag_list = sorted(self.tags)

    def trans_logprob(self, t, t1, t2):
        """P(t | t2, t1)."""
        num = self.trans_tri[(t2, t1, t)] + self.k_trans
        den = self.trans_bi[(t2, t1)] + self.k_trans * self.T
        return math.log(num / den)

    def emit_logprob(self, w, t):
        """P(w | t), with a uniform-ish fallback for unseen words."""
        if self.word_count[w] == 0:
            return math.log(self.k_emit / (self.tag_count[t] + self.k_emit * self.T))
        num = self.emit[(t, w)] + self.k_emit
        den = self.tag_count[t] + self.k_emit * len(self.word_count)
        return math.log(num / den)

    def viterbi_tag(self, words, beam_size=100):
        """Second-order (trigram) Viterbi over tag-pair states.
        dp[i][(t_{i-1}, t_i)] = (best_score, prev_state).

        BUGFIX / performance: the original version enumerated *every* tag-pair state x every
        tag at every position (O(n * T^2) per sentence with T = number of distinct tags). For
        plain English POS tags (T ~= 12) that's fine, but for the morphology-aware Spanish
        tagset (T can be 60-100+ once tags are expanded into NOUN-Fem-Sg / NOUN-Masc-Pl / ...
        combinations) that quadratic blow-up made tagging thousands of dev/test sentences
        effectively hang (this is what triggered the KeyboardInterrupt in the earlier run).
        We keep the algorithm exact for small tagsets but cap the number of surviving states
        carried forward at each position to `beam_size` (keeping only the highest-scoring
        ones), which is a standard, safe beam-search pruning for Viterbi POS tagging and keeps
        runtime roughly linear in beam_size instead of T^2."""
        n = len(words)
        if n == 0:
            return []
        tags = self.tag_list
        dp = [dict() for _ in range(n + 1)]
        dp[0][(BOS, BOS)] = (0.0, None)
        for i in range(1, n + 1):
            w = words[i - 1]
            emit_cache = {t: self.emit_logprob(w, t) for t in tags}
            prev_states = dp[i - 1]
            if beam_size is not None and len(prev_states) > beam_size:
                prev_states = dict(sorted(prev_states.items(), key=lambda kv: kv[1][0],
                                           reverse=True)[:beam_size])
            for (t_im2, t_im1), (score, _prev) in prev_states.items():
                for t in tags:
                    lp = score + self.trans_logprob(t, t_im1, t_im2) + emit_cache[t]
                    state = (t_im1, t)
                    if state not in dp[i] or lp > dp[i][state][0]:
                        dp[i][state] = (lp, (t_im2, t_im1))
            if beam_size is not None and len(dp[i]) > beam_size:
                dp[i] = dict(sorted(dp[i].items(), key=lambda kv: kv[1][0],
                                     reverse=True)[:beam_size])
        if not dp[n]:
            return [self.tag_list[0]] * n
        best_state = max(dp[n], key=lambda s: dp[n][s][0])
        tags_out, state, i = [], best_state, n
        while i > 0:
            score, prev_state = dp[i][state]
            tags_out.append(state[1])
            state = prev_state
            i -= 1
        tags_out.reverse()
        return tags_out


def most_frequent_tag_baseline(train_tagged_sentences):
    """Returns (word -> most-frequent-tag dict, default_tag for unseen words)."""
    counts = defaultdict(Counter)
    overall = Counter()
    for sent in train_tagged_sentences:
        for w, t in sent:
            counts[w][t] += 1
            overall[t] += 1
    word2tag = {w: c.most_common(1)[0][0] for w, c in counts.items()}
    default_tag = overall.most_common(1)[0][0]
    return word2tag, default_tag


In [26]:
# quick sanity check
_hmm = TrigramHMM(0.1, 0.05); _hmm.train(ENGLISH_FALLBACK * 20)
_words = ["the","quick","brown","fox","jumps","over","the","lazy","dog"]
print(list(zip(_words, _hmm.viterbi_tag(_words))))


[('the', 'DET'), ('quick', 'ADJ'), ('brown', 'ADJ'), ('fox', 'NOUN'), ('jumps', 'VERB'), ('over', 'ADP'), ('the', 'DET'), ('lazy', 'ADJ'), ('dog', 'NOUN')]


## 6. Part 5 — Evaluation: accuracy, confusion matrix, error-source breakdown

For each test sentence we:
1. Remove the spaces to get the raw character string (this simulates the assignment's input).
2. Segment it (trigram+DP, or the greedy baseline).
3. Tag the resulting words (trigram HMM, or the most-frequent-tag baseline).
4. Compare predicted `(word, tag)` spans to the gold spans.

For the **error-source breakdown**, a predicted word only contributes a *taggable* comparison if
its character span exactly matches a gold word's span:
- span **mismatch** → counted as a **segmentation-caused error** (its tag is automatically wrong,
  as in the assignment's `thequick|brown|fox` example).
- span **match**, tag **mismatch** → counted as a **genuine tagging error**.
- span **match**, tag **match** → correct.

The confusion matrix is built only over the "span match" subset, so it's a fair like-for-like
comparison of tags (comparing tags for two different words at mismatched spans wouldn't be
meaningful).

In [27]:
def spans_from_words(words):
    spans, pos = [], 0
    for w in words:
        spans.append((pos, pos + len(w)))
        pos += len(w)
    return spans


def segmentation_accuracy(pred_words, gold_words):
    """Fraction of gold word spans recovered exactly by the predicted segmentation."""
    gold_spans = set(spans_from_words(gold_words))
    pred_spans = set(spans_from_words(pred_words))
    if not gold_spans:
        return 1.0
    return len(gold_spans & pred_spans) / len(gold_spans)


def evaluate_sentence(pred_words, pred_tags, gold_words, gold_tags):
    gold_spans = spans_from_words(gold_words)
    pred_span_to_word = {sp: (w, t) for sp, w, t in
                          zip(spans_from_words(pred_words), pred_words, pred_tags)}
    n_seg_err = n_genuine_err = n_correct = 0
    pairs = []
    for sp, gw, gt in zip(gold_spans, gold_words, gold_tags):
        if sp not in pred_span_to_word:
            n_seg_err += 1
            continue
        pw, pt = pred_span_to_word[sp]
        pairs.append((gt, pt))
        if pt == gt:
            n_correct += 1
        else:
            n_genuine_err += 1
    return {"n_gold_words": len(gold_words), "n_correct": n_correct,
            "n_segmentation_errors": n_seg_err, "n_genuine_tag_errors": n_genuine_err,
            "pairs": pairs}


def run_full_evaluation(test_sentences, seg_lm, seg_vocab, hmm, w2t_baseline, default_tag,
                         max_word_len=15, use_baselines=False):
    totals = Counter()
    all_pairs = []
    seg_acc_sum = 0.0
    for sent in test_sentences:
        gold_words = [w for w, t in sent]
        gold_tags = [t for w, t in sent]
        char_string = "".join(gold_words)

        if use_baselines:
            pred_words = greedy_longest_match_segment(char_string, seg_vocab, max_word_len)
            pred_tags = [w2t_baseline.get(w, default_tag) for w in pred_words]
        else:
            pred_words = viterbi_segment(char_string, seg_lm, seg_vocab, max_word_len)
            pred_tags = hmm.viterbi_tag(pred_words)

        seg_acc_sum += segmentation_accuracy(pred_words, gold_words)
        res = evaluate_sentence(pred_words, pred_tags, gold_words, gold_tags)
        for k in ("n_gold_words", "n_correct", "n_segmentation_errors", "n_genuine_tag_errors"):
            totals[k] += res[k]
        all_pairs.extend(res["pairs"])

    n_sent = max(len(test_sentences), 1)
    return {
        "segmentation_accuracy": seg_acc_sum / n_sent,
        "tagging_accuracy_overall": totals["n_correct"] / max(totals["n_gold_words"], 1),
        "tagging_accuracy_given_correct_seg":
            totals["n_correct"] / max(totals["n_correct"] + totals["n_genuine_tag_errors"], 1),
        "n_segmentation_caused_errors": totals["n_segmentation_errors"],
        "n_genuine_tagging_errors": totals["n_genuine_tag_errors"],
        "n_correct": totals["n_correct"],
        "n_gold_words": totals["n_gold_words"],
        "pairs": all_pairs,
    }


def confusion_matrix_report(pairs):
    from sklearn.metrics import confusion_matrix
    if not pairs:
        return [], []
    gold = [g for g, p in pairs]
    pred = [p for g, p in pairs]
    labels = sorted(set(gold) | set(pred))
    return labels, confusion_matrix(gold, pred, labels=labels)


def print_confusion_matrix(labels, cm):
    header = "        " + " ".join(f"{l[:6]:>6}" for l in labels)
    print(header)
    for i, l in enumerate(labels):
        row = " ".join(f"{cm[i][j]:>6}" for j in range(len(labels)))
        print(f"{l[:6]:>6}  {row}")


## 7. English pipeline: train/test split, training, sample output, evaluation

In [28]:
english_data = load_english_data()
random.shuffle(english_data)
split = int(len(english_data) * 0.8)
en_train, en_test = english_data[:split], english_data[split:]
if not en_test:
    en_test = en_train[-2:]

en_words_sents = [[w for w, t in s] for s in en_train]
en_seg_lm = TrigramLM(k=0.3); en_seg_lm.train(en_words_sents)
en_seg_vocab = set(w for s in en_words_sents for w in s)

en_hmm = TrigramHMM(k_trans=0.1, k_emit=0.05); en_hmm.train(en_train)
en_w2t, en_default = most_frequent_tag_baseline(en_train)

print(f"English: {len(en_train)} train sentences, {len(en_test)} test sentences")


[english] loaded 57340 sentences from NLTK Brown corpus
English: 45872 train sentences, 11468 test sentences


In [29]:
# --- sample test strings from the assignment ---
for s in ["thequickbrownfoxjumpsoverthelazydog", "thequickbrownfox"]:
    pw = viterbi_segment(s, en_seg_lm, en_seg_vocab, max_word_len=12)
    pt = en_hmm.viterbi_tag(pw)
    print(s, "->", list(zip(pw, pt)))


thequickbrownfoxjumpsoverthelazydog -> [('the', 'DET'), ('quick', 'ADJ'), ('brown', 'ADJ'), ('fox', 'NOUN'), ('jumps', 'VERB'), ('overt', 'ADJ'), ('he', 'PRON'), ('lazy', 'ADJ'), ('dog', 'NOUN')]
thequickbrownfox -> [('the', 'DET'), ('quick', 'ADJ'), ('brown', 'ADJ'), ('fox', 'NOUN')]


In [30]:
en_report_model = run_full_evaluation(en_test, en_seg_lm, en_seg_vocab, en_hmm,
                                       en_w2t, en_default, max_word_len=12, use_baselines=False)
en_report_base = run_full_evaluation(en_test, en_seg_lm, en_seg_vocab, en_hmm,
                                      en_w2t, en_default, max_word_len=12, use_baselines=True)

print("=== English: trigram+DP model ===")
for k, v in en_report_model.items():
    if k != "pairs":
        print(f"  {k}: {v}")

print("\n=== English: baselines (greedy-longest-match + most-frequent-tag) ===")
for k, v in en_report_base.items():
    if k != "pairs":
        print(f"  {k}: {v}")

labels, cm = confusion_matrix_report(en_report_model["pairs"])
print("\nConfusion matrix (model, gold=rows, pred=cols):")
print_confusion_matrix(labels, cm)


=== English: trigram+DP model ===
  segmentation_accuracy: 0.8650309607110492
  tagging_accuracy_overall: 0.8384034594296592
  tagging_accuracy_given_correct_seg: 0.9633101072884912
  n_segmentation_caused_errors: 30105
  n_genuine_tagging_errors: 7414
  n_correct: 194658
  n_gold_words: 232177

=== English: baselines (greedy-longest-match + most-frequent-tag) ===
  segmentation_accuracy: 0.7827557377709682
  tagging_accuracy_overall: 0.7303479672835811
  tagging_accuracy_given_correct_seg: 0.9529027653679946
  n_segmentation_caused_errors: 54226
  n_genuine_tagging_errors: 8381
  n_correct: 169570
  n_gold_words: 232177

Confusion matrix (model, gold=rows, pred=cols):
             .    ADJ    ADP    ADV   CONJ    DET   NOUN    NUM   PRON    PRT   VERB      X
     .   23857      0      0      0      0      0      0      0      0      0      0      6
   ADJ       2  11856     14    156      8     99    277      5      3      9     75     82
   ADP       0      6  20950     99    180    

## 8. Spanish pipeline (plain UPOS tags vs. morphology-aware tags)

In [31]:
spanish_raw = load_spanish_data()  # list of (form, upos, feats) sentences
random.shuffle(spanish_raw)
split = int(len(spanish_raw) * 0.8)
es_train_raw, es_test_raw = spanish_raw[:split], spanish_raw[split:]
if not es_test_raw:
    es_test_raw = es_train_raw[-2:]

es_train_plain = spanish_to_tagged(es_train_raw, morph=False)
es_test_plain  = spanish_to_tagged(es_test_raw,  morph=False)
es_train_morph = spanish_to_tagged(es_train_raw, morph=True)
es_test_morph  = spanish_to_tagged(es_test_raw,  morph=True)

es_words_sents = [[w for w, t in s] for s in es_train_plain]
es_seg_lm = TrigramLM(k=0.3); es_seg_lm.train(es_words_sents)
es_seg_vocab = set(w for s in es_words_sents for w in s)

es_hmm_plain = TrigramHMM(0.1, 0.05); es_hmm_plain.train(es_train_plain)
es_hmm_morph = TrigramHMM(0.1, 0.05); es_hmm_morph.train(es_train_morph)
es_w2t, es_default = most_frequent_tag_baseline(es_train_plain)

print(f"Spanish: {len(es_train_plain)} train sentences, {len(es_test_plain)} test sentences")


[spanish] loaded 14186 sentences from ./UD_Spanish-GSD/es_gsd-ud-train.conllu
Spanish: 11348 train sentences, 2838 test sentences


In [32]:
# --- sample test strings from the assignment ---
for s in ["mispadrespuedenviajar", "elcielodespejadoesazul", "lacasarojaesgrande"]:
    pw = viterbi_segment(s, es_seg_lm, es_seg_vocab, max_word_len=12)
    print(s, "-> plain:", list(zip(pw, es_hmm_plain.viterbi_tag(pw))))
    print(" " * len(s), "-> morph:", list(zip(pw, es_hmm_morph.viterbi_tag(pw))))


mispadrespuedenviajar -> plain: [('mis', 'DET'), ('padres', 'NOUN'), ('pueden', 'AUX'), ('viajar', 'VERB')]
                      -> morph: [('mis', 'DET-Pl'), ('padres', 'NOUN-Masc-Pl'), ('pueden', 'AUX-Pl'), ('viajar', 'VERB')]
elcielodespejadoesazul -> plain: [('el', 'DET'), ('cielo', 'NOUN'), ('des', 'ADP'), ('pejadoes', 'DET'), ('azul', 'NOUN')]
                       -> morph: [('el', 'DET-Masc-Sg'), ('cielo', 'NOUN-Masc-Sg'), ('des', 'X'), ('pejadoes', 'X-Masc'), ('azul', 'ADJ-Sg')]
lacasarojaesgrande -> plain: [('la', 'DET'), ('casa', 'NOUN'), ('roja', 'ADJ'), ('es', 'AUX'), ('grande', 'ADJ')]
                   -> morph: [('la', 'DET-Fem-Sg'), ('casa', 'NOUN-Fem-Sg'), ('roja', 'ADJ-Fem-Sg'), ('es', 'AUX-Sg'), ('grande', 'ADJ-Sg')]


In [33]:
es_report_plain = run_full_evaluation(es_test_plain, es_seg_lm, es_seg_vocab, es_hmm_plain,
                                       es_w2t, es_default, max_word_len=12, use_baselines=False)
es_report_morph = run_full_evaluation(es_test_morph, es_seg_lm, es_seg_vocab, es_hmm_morph,
                                       es_w2t, es_default, max_word_len=12, use_baselines=False)
es_report_base  = run_full_evaluation(es_test_plain, es_seg_lm, es_seg_vocab, es_hmm_plain,
                                       es_w2t, es_default, max_word_len=12, use_baselines=True)

print("=== Spanish: plain-tag model ===")
for k, v in es_report_plain.items():
    if k != "pairs": print(f"  {k}: {v}")

print("\n=== Spanish: morphology-aware-tag model ===")
for k, v in es_report_morph.items():
    if k != "pairs": print(f"  {k}: {v}")

print("\n=== Spanish: baselines ===")
for k, v in es_report_base.items():
    if k != "pairs": print(f"  {k}: {v}")

labels_es, cm_es = confusion_matrix_report(es_report_plain["pairs"])
print("\nConfusion matrix (plain-tag model):")
print_confusion_matrix(labels_es, cm_es)


=== Spanish: plain-tag model ===
  segmentation_accuracy: 0.7371827541970811
  tagging_accuracy_overall: 0.6761197950873505
  tagging_accuracy_given_correct_seg: 0.9232328305203308
  n_segmentation_caused_errors: 20377
  n_genuine_tagging_errors: 4280
  n_correct: 51473
  n_gold_words: 76130

=== Spanish: morphology-aware-tag model ===
  segmentation_accuracy: 0.7371827541970811
  tagging_accuracy_overall: 0.6621174307106266
  tagging_accuracy_given_correct_seg: 0.9041127831686188
  n_segmentation_caused_errors: 20377
  n_genuine_tagging_errors: 5346
  n_correct: 50407
  n_gold_words: 76130

=== Spanish: baselines ===
  segmentation_accuracy: 0.6403304787521158
  tagging_accuracy_overall: 0.5960593721266255
  tagging_accuracy_given_correct_seg: 0.9346267918932278
  n_segmentation_caused_errors: 27578
  n_genuine_tagging_errors: 3174
  n_correct: 45378
  n_gold_words: 76130

Confusion matrix (plain-tag model):
           ADJ    ADP    ADV    AUX  CCONJ    DET   INTJ   NOUN    NUM   PART

## 9. Comparative report

*(The numbers referenced below are computed live from the cells above — re-run the notebook with
the real Brown / UD-Spanish-GSD corpora loaded for a meaningful, publishable comparison. With the
tiny embedded fallback corpus the models can trivially memorise the data, so accuracies will look
artificially perfect; that's expected and is exactly why the assignment asks you to use the full
corpora with a proper train/dev/test split.)**

In [34]:
print("English  — trigram+DP tagging accuracy:", en_report_model["tagging_accuracy_overall"],
      " | baseline:", en_report_base["tagging_accuracy_overall"])
print("Spanish  — plain-tag accuracy:", es_report_plain["tagging_accuracy_overall"],
      " | morph-tag accuracy:", es_report_morph["tagging_accuracy_overall"],
      " | baseline:", es_report_base["tagging_accuracy_overall"])
print()
print("English segmentation-caused vs genuine tagging errors:",
      en_report_model["n_segmentation_caused_errors"], "vs", en_report_model["n_genuine_tagging_errors"])
print("Spanish segmentation-caused vs genuine tagging errors:",
      es_report_plain["n_segmentation_caused_errors"], "vs", es_report_plain["n_genuine_tagging_errors"])


English  — trigram+DP tagging accuracy: 0.8384034594296592  | baseline: 0.7303479672835811
Spanish  — plain-tag accuracy: 0.6761197950873505  | morph-tag accuracy: 0.6621174307106266  | baseline: 0.5960593721266255

English segmentation-caused vs genuine tagging errors: 30105 vs 7414
Spanish segmentation-caused vs genuine tagging errors: 20377 vs 4280


**Where did English and Spanish differ most in accuracy?**
Spanish carries grammatical gender/number agreement that English lacks, so a plain trigram HMM on
Spanish UPOS tags has to lean more heavily on lexical (emission) evidence, while the morphology-aware
tagset gives the transition model an explicit agreement signal (e.g. `ADJ-Fem-Sg` following
`NOUN-Fem-Sg`). On a real UD-Spanish-GSD run, expect the *plain* Spanish tagger to underperform the
English tagger slightly (richer inflection → more distinct word forms → more unseen/rare words in
the emission model), and the *morphology-aware* Spanish tagger to close part of that gap on
agreement-sensitive positions (adjectives, determiners) while occasionally adding **more** tag
categories to confuse when data is sparse (fewer training examples per fine-grained tag).

**Did agreement-aware tagging help, or add noise?**
It helps in principle by supplying agreement structure, but it also fragments each POS class into
several finer-grained tags (`NOUN-Fem-Sg`, `NOUN-Fem-Pl`, `NOUN-Masc-Sg`, ...), which means each
gets fewer training examples. Whether it nets positive depends on corpus size — on the full
UD-Spanish-GSD training split there should be enough data for morphology-aware tags to help; on a
tiny corpus (like the embedded fallback here) the sparsity effect can dominate. Compare
`es_report_plain["tagging_accuracy_overall"]` vs `es_report_morph["tagging_accuracy_overall"]`
above on your real run.

**How much of the tagging error came from segmentation vs. genuine tagging mistakes?**
The `n_segmentation_caused_errors` vs `n_genuine_tagging_errors` counts above answer this directly
per language. Segmentation-caused errors dominate when the trigram+DP segmenter is undertrained
(small vocabulary/LM) or when the test set contains many words absent from the training vocabulary;
genuine tagging errors dominate once segmentation is essentially solved, and tend to cluster on
ambiguous open-class words (e.g. ADJ/NOUN confusions) as seen in the confusion matrices above.

**How much better were the trigram+DP models than the simple baselines?**
`en_report_base` / `es_report_base` above show the greedy-longest-match + most-frequent-tag
baseline's numbers. The trigram+DP segmenter should out-perform greedy longest-match specifically
on ambiguous splits (where the longest greedy match is not the linguistically correct one — e.g.
"thequick" swallowing into one long known word instead of splitting at the right boundary), and the
trigram HMM should out-perform most-frequent-tag on context-dependent ambiguous words (the same
word type taking different tags depending on its neighbours).

## 10. Notes / how to extend to German instead of Spanish

Everything above is language-agnostic except `spanish_to_tagged` (which is trivial — it's just a
thin wrapper renaming the (form, upos, feats) tuples). To run German instead:

```python
german_raw = load_conllu_corpus('./UD_German-GSD/de_gsd-ud')  # write an analogous loader
german_train_plain = spanish_to_tagged(german_raw, morph=False)  # function name is generic despite the name
german_train_morph = spanish_to_tagged(german_raw, morph=True)   # German also marks Gender/Number/Case
```

German additionally marks **Case**, which you may want to fold into `build_morph_tag` alongside
Gender/Number for a richer agreement-aware tagset, since German adjective/article endings agree in
gender, number, *and* case.